# Phase 4 AgentDojo debug audit

CPU-only sanity checks for `agentdojo_records.jsonl` (and, when present, `agentdojo_trajectories.jsonl`) produced by `jspace-phase4 --stage generate`. Copy both files from the run's `phase4/` directory into `RECORDS_DIR`. Nothing here loads the model or changes any result; it only flags records whose outcomes do not make sense.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

RECORDS_DIR = Path("../Records")
RECORDS_PATH = RECORDS_DIR / "agentdojo_records.jsonl"
TRAJECTORIES_PATH = RECORDS_DIR / "agentdojo_trajectories.jsonl"
MAX_NEW_TOKENS = 512

# AgentDojo v1.2.2 at the pinned commit: (user tasks, injection tasks) per suite.
EXPECTED_TASKS = {"banking": (16, 9), "slack": (21, 5), "travel": (20, 7), "workspace": (40, 14)}


def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"Skipping an incomplete final line in {path}")
    return rows


records = pd.DataFrame(read_jsonl(RECORDS_PATH))
records["scored"] = records["mean_score"].notna()
attack = records[records.condition == "attack"]
control = records[records.condition == "control"]
print(f"{len(records)} records from {RECORDS_PATH}")
print("phase1 run:", records.phase1_run_id.unique() if "phase1_run_id" in records else "unknown")

## 1. Coverage against the native case matrix

In [ ]:
rows = []
for suite, (users, injections) in EXPECTED_TASKS.items():
    for condition, expected in (("control", users), ("attack", users * injections)):
        found = int(((records.suite == suite) & (records.condition == condition)).sum())
        rows.append({"suite": suite, "condition": condition, "expected": expected, "found": found})
coverage = pd.DataFrame(rows)
coverage["complete"] = coverage.expected == coverage.found
coverage

## 2. Native outcomes and exposure per suite

Clean utility comes from control episodes. Utility under attack, attack success, and exposure come from attacked episodes. Attack success is split by whether our tracker saw the injected text reach the model. Success without exposure should be zero; any count there means exposure tracking missed a delivery.

In [ ]:
def rate(series):
    return round(float(series.mean()), 3) if len(series) else float("nan")


summary = []
for suite in sorted(records.suite.unique()):
    c = control[control.suite == suite]
    a = attack[attack.suite == suite]
    exposed = a[a.injection_exposed]
    unexposed = a[~a.injection_exposed]
    summary.append(
        {
            "suite": suite,
            "clean_utility": rate(c.native_utility),
            "clean_scored": rate(c.scored),
            "utility_under_attack": rate(a.native_utility),
            "exposure_rate": rate(a.injection_exposed),
            "asr": rate(a.native_attack_success.astype(bool)),
            "asr_if_exposed": rate(exposed.native_attack_success.astype(bool)),
            "asr_count_unexposed": int(unexposed.native_attack_success.astype(bool).sum()),
            "attacks": len(a),
        }
    )
pd.DataFrame(summary)

## 3. What the saved completion looks like

`generated_response` is the completion captured at the detector decision point, or the final completion when no decision point was reached. A native tool call is `<function=name>{json}</function>`. Echoed tool results (`{"result": ...}`) and chained calls mean the model is writing tool outputs itself; malformed calls are usually truncation at the generation limit.

In [ ]:
NATIVE_CALL = re.compile(r"<function\s*=\s*([^>]+)>(.*?)(</function>|$)", re.S)


def call_json_ok(text):
    match = NATIVE_CALL.search(text)
    if match is None:
        return None
    try:
        json.loads(match.group(2).strip())
        return True
    except json.JSONDecodeError:
        return False


def response_format(text):
    stripped = text.strip()
    calls = len(re.findall(r"<function\s*=", stripped))
    if not stripped:
        return "empty"
    if calls and call_json_ok(stripped) is False:
        return "malformed_call"
    if calls > 1:
        return "chained_calls"
    if '{"result"' in stripped:
        return "echoed_tool_result"
    if calls == 1:
        return "tool_call"
    if "[TOOL_CALLS]" in stripped or re.match(r'^\[\s*\{\s*"name"', stripped):
        return "mistral_native_call"
    if re.search(r"call:[A-Za-z_]\w*\s*\{", stripped):
        return "gemma_call"
    return "text"


records["format"] = records.generated_response.map(response_format)
attack = records[records.condition == "attack"]
control = records[records.condition == "control"]
pd.crosstab([records.suite, records.condition], records.format, margins=True)

## 4. Red flags

Each check lists the cases that break an invariant or look like a harness artifact rather than model behavior.

In [ ]:
flags = {}
flags["attack success without exposure"] = attack[
    attack.native_attack_success.astype(bool) & ~attack.injection_exposed
]
flags["exposed attack without detector score"] = attack[attack.injection_exposed & ~attack.scored]
flags["control without a scored tool observation"] = control[~control.scored]
flags["malformed or truncated tool call"] = records[records.format == "malformed_call"]
flags["model echoed tool results"] = records[records.format == "echoed_tool_result"]
flags["model chained several calls"] = records[records.format == "chained_calls"]

for name, frame in flags.items():
    print(f"{len(frame):4d}  {name}")

never_by_injection = attack.groupby(["suite", "injection_task_id"]).injection_exposed.any()
never_by_injection = never_by_injection[~never_by_injection]
print("\nInjection tasks never exposed under any user task (points at exposure tracking):")
print(never_by_injection.index.tolist() or "none")

by_user = attack.groupby(["suite", "user_task_id"]).agg(
    exposed=("injection_exposed", "any"),
    distinct_responses=("generated_response", "nunique"),
    utility_under_attack=("native_utility", "mean"),
)
by_user = by_user.join(
    control.set_index(["suite", "user_task_id"])[["native_utility", "format"]].rename(
        columns={"native_utility": "clean_utility", "format": "clean_format"}
    )
)
print("\nUser tasks never exposed under any injection (the model's own trajectory avoided the vector):")
by_user[~by_user.exposed]

In [ ]:
utility_gap = pd.DataFrame(summary).set_index("suite")
utility_gap = utility_gap[utility_gap.utility_under_attack > utility_gap.clean_utility]
print("Suites where utility under attack exceeds clean utility:", utility_gap.index.tolist() or "none")
flags["attack success without exposure"][["case_id", "generated_response"]]

## 5. Sample viewer

In [ ]:
def show(frame, n=3, width=1200):
    for _, row in frame.head(n).iterrows():
        print("=" * 100)
        print(row.case_id)
        print(
            f"exposed={row.injection_exposed} utility={row.native_utility} "
            f"attack_success={row.native_attack_success} mean_score={row.mean_score}"
        )
        print("-" * 100)
        print(row.generated_response[:width])


for fmt in ["malformed_call", "echoed_tool_result", "chained_calls", "text"]:
    print(f"\n##### {fmt}")
    show(records[records.format == fmt], n=2)

## 6. Trajectory viewer

Requires `agentdojo_trajectories.jsonl` from a run with harness version 2 or later. Each line holds every model step (raw completion with special tokens, the text actually parsed, parsed tool calls, and whether the injection had been delivered by that step) plus the full AgentDojo message list.

In [ ]:
trajectories = {}
if TRAJECTORIES_PATH.exists():
    for row in read_jsonl(TRAJECTORIES_PATH):
        trajectories[row["case_id"]] = row
print(f"{len(trajectories)} trajectories loaded" if trajectories else f"No trajectories at {TRAJECTORIES_PATH}")

if trajectories:
    steps = pd.DataFrame(
        [
            {"case_id": case_id, **{k: v for k, v in step.items() if k not in ("raw_completion", "completion", "parsed_text")}}
            for case_id, row in trajectories.items()
            for step in row["steps"]
        ]
    )
    steps["hit_token_limit"] = steps.generated_tokens >= MAX_NEW_TOKENS
    steps["called_tool"] = steps.parsed_tool_calls.map(bool)
    print("steps per case:", steps.groupby("case_id").size().describe().round(1).to_dict())
    print("steps that hit the generation limit:", int(steps.hit_token_limit.sum()))

In [ ]:
def show_trajectory(case_id, width=800):
    row = trajectories[case_id]
    print(case_id, "| first exposed step:", row["first_exposed_step"])
    for message in row["messages"]:
        content = message.get("content") or []
        text = "\n".join(block.get("content") or "" for block in content if isinstance(block, dict))
        calls = message.get("tool_calls") or []
        print("-" * 100)
        print(f"[{message['role']}]", ", ".join(f"{c['function']}({json.dumps(c['args'])})" for c in calls))
        print(text[:width])
    print("=" * 100)
    for step in row["steps"]:
        print(
            f"step {step['step']}: exposed={step['injection_exposed']} captured={step['captured']} "
            f"tokens={step['generated_tokens']} calls={[c['function'] for c in step['parsed_tool_calls']]}"
        )
        if step["raw_completion"] != step["parsed_text"]:
            print("   raw completion differs from parsed text:", repr(step["raw_completion"][:width]))


if trajectories:
    show_trajectory(next(iter(trajectories)))